In [ ]:
import sys
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, FloatType, LongType
from pyspark.sql.functions import col, lit, year, month, to_date, to_timestamp

spark = SparkSession.builder\
        .appName("e-commerce_bronze_transform")\
        .master("local[*]")\
        .config("spark.executor.memory", "2g")\
        .config("spark.driver.memory", "2g")\
        .getOrCreate()

In [ ]:
# Adjust this to your actual project root
project_root = "/home/lpascual/Projects/E-Commerce_DWH/scripts"

if project_root not in sys.path:
    sys.path.append(project_root)

In [ ]:
from infrastructure.schemas.glue.datalake_schemas import bronze_schemas

In [ ]:
data_path = "/home/lpascual/Projects/E-Commerce_DWH/data/bronze/landing/2019-Oct-enriched.csv"
output_dir = "/home/lpascual/Projects/E-Commerce_DWH/data/bronze/parquet/"

TODO List
1) Schema Enforcement (Evolution)
2) Data Quality Checks
    * Confirm email is in correct format \w@\w
    * Price is > 0
    * User Session is in \w{8}-\w{4}-\w{4}-\w{4}-\w{12} format
    * If category_code is not NULL, confirm the value is in ^\w+\.?(\w+\.)?(\w+)?$
    * Event Time is not Null

In [ ]:
df = spark.read.format("csv")\
    .schema(bronze_schemas["bronze"])\
    .option("header", "true")\
    .load(data_path)
print("Number of partitions:", df.rdd.getNumPartitions())

In [ ]:
df = df.withColumn("event_ts", to_timestamp(col("event_time"), "yyyy-MM-dd HH:mm:ss 'UTC'")) \
       .withColumn("date", to_date(col("event_ts"))) \
       .withColumn("ingestion_ts", lit(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))) \
       .withColumn("input_filename", lit("2019-Oct-enriched.csv"))

# Extract year and month
df = df.withColumn("year", year(col("date"))) \
       .withColumn("month", month(col("date")))

In [ ]:
df.show(truncate=False)

In [ ]:
df.write.partitionBy("year", "month").parquet(path=output_dir, mode="overwrite")

In [ ]:
spark.stop()